# Colab preparation

This cell gets the latest code from GitHub (clone on first run, then `git fetch` + `reset --hard origin/main`), mounts Google Drive, and installs only missing packages. It does not train or draw the figure.

Run this cell after we push updates; it prints the repo commit hash. Colab can still *display* old markdown in cells you opened earlier—trust the printed commit, or reopen the notebook from GitHub. Data: `My Drive/Foundation Model/vendor/pfdelta_task1.1`.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Repo at {head}", flush=True)
    from google.colab import drive

    drive.mount("/content/drive")

missing = []
for module, package in (
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("pyarrow", "pyarrow"),
    ("huggingface_hub", "huggingface_hub"),
):
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Required packages are already installed.")


# Training

This cell trains **TPGT** only (type-pair graph Transformer, independent weights per stage) on that Drive dataset. It writes checkpoints to `Foundation Model/tpgt_task11_v7` and comparison tables to `Foundation Model/results/tpgt_task11_v7` (the folder name follows `RUN_TAG` in the script, so older runs such as `Foundation Model/tpgt_task11` are never overwritten). It does not draw the figure.

Use a **GPU** runtime. Run the prep cell first so you have the latest `train_tpgt_task11.py` from GitHub. A finished seed (100 epochs, matching checkpoint version) is skipped if this cell is run again; obsolete Drive checkpoints (wrong version, old RPE width, or corrupt files) are ignored and training restarts from epoch 0.

Training uses feasible **N** only (same split as GENCO). **100 epochs**, batch **32**, loss **0.85** state + **0.15** physics, seeds 0–2. Set **`DATA_FRACTION`** in the training cell (e.g. **0.25**) to use the **first** that fraction of scenarios in each train/valid/test folder (fixed parquet order). Raising it to **0.30** keeps the same first 25% and **adds** the next 5%; cached splits extend instead of rebuilding from scratch. **1.0** is the full dataset.

Optional: set `GENCO_TPGT_COMPILE=1` (or pass `--compile` / `--fast` to `%run`) for `torch.compile`; default training already uses batched type-pair attention.

Pair features are a 5-channel **meshed-network RPE** (full topology: \(\mathrm{Re}\,Y_{ij}\), \(\mathrm{Im}\,Y_{ij}\), adjacency, hop distance, effective resistance), not a spanning tree. During training each RPE channel is zeroed with probability 0.1 per step (never at validation/test). Width/depth (48 hidden, 24 pair/attn, 2 heads, 4 correction stages, 3-layer MLPs) are implementation choices, not the authors' hyperparameters.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# 1.0 = full PFΔ split; 0.25 ≈ 12150 train / 1350 valid / 500 test (N) cases. Batch size is always 32. Restart runtime if GPU OOM.
DATA_FRACTION = 1.0
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"DATA_FRACTION={DATA_FRACTION}", flush=True)

%run train_tpgt_task11.py


# Diagnostics and bus error maps

Run these after (or between) training sessions; they cannot run while the training cell is running. Both use the latest checkpoint of the current run folder (`Foundation Model/tpgt_task11_v7`) and the split caches the training cell built.

**Diagnose** prints: oracle / flat-start / per-stage PBL, state RMSE by bus type, RPE channel zero-ablation, RPE and residual correctness audits, and the N-1 / N-2 contingency report split into line vs generator outages.

**Bus error maps** draw per-bus PBL on the IEEE 118 one-line layout (validation N, worst generator outage, worst and median line outage), plus a hop-distance table, and display them below.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
os.chdir(REPO)
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive

    drive.mount("/content/drive")
os.environ["GENCO_PFDELTA_TASK11"] = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"

%run train_tpgt_task11.py --diagnose --seed 0 --cases 32


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

os.chdir("/content/genco-figure4-task11")
RUN = Path("/content/drive/MyDrive/Foundation Model/tpgt_task11_v7")
SEED = 0
CHECKPOINT = RUN / f"seed_{SEED}" / "last.pt"
CACHE = Path("/content/tpgt_task11_v7_cache")
if not (CACHE / "test_n-1_feasible.pt").is_file():
    CACHE = RUN / "cache"
OUT = RUN / "bus_error_maps" / f"seed_{SEED}"

%run plot_bus_error_map.py --checkpoint "{CHECKPOINT}" --cache-dir "{CACHE}" --out-dir "{OUT}"

for name in (
    "val_N_mean_pbl.png",
    "val_N_dP_dQ.png",
    "test_N1_gen_outage_worst.png",
    "test_N1_line_outage_worst.png",
    "test_N1_line_outage_median.png",
):
    display(Image(filename=str(OUT / name)))


# SVG figure

This cell draws the published Task 1.1 SVG (GENCO Base, CANOS-PF, GNS-S, PFNet) from the published table. It does not train.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


here = Path.cwd()
if not (here / "plot_task1_1.py").is_file():
    if on_colab():
        if not REPO.is_dir():
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
        os.chdir(REPO)
    elif (here / "genco" / "plot_task1_1.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("plot_task1_1.py is not in this folder.")

%run plot_task1_1.py --published
